In [0]:
bronze_county = spark.table(
    "workforce_analytics.bronze.dim_county_raw"
)

bronze_county.printSchema()

display(bronze_county.limit(10))

Step 1: Create the cleaned DimCounty table

In [0]:
from pyspark.sql import functions as F

# Read Bronze table
bronze_county = spark.table(
    "workforce_analytics.bronze.dim_county_raw"
)

# Clean the data
silver_county = bronze_county.select(
    F.trim(F.col("county_id")).alias("county_id"),
    F.trim(F.col("county_name")).alias("county_name"),
    F.trim(F.col("region")).alias("region"),
    F.trim(F.col("country")).alias("country"),
    F.trim(F.col("population_band")).alias("population_band"),
    F.trim(F.col("urban_rural")).alias("urban_rural"),
    F.col("_source_file"),
    F.col("_ingested_at")
)

Step 2: Validate the data

In [0]:
# Check schema and sample records
silver_county.printSchema()

display(silver_county)

# Check total rows
print("Total rows:", silver_county.count())

# Check missing county IDs
print(
    "Missing county IDs:",
    silver_county.filter(
        F.col("county_id").isNull() | (F.col("county_id") == "")
    ).count()
)

# Check duplicate county IDs
duplicate_counties = (
    silver_county
    .groupBy("county_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate county IDs:", duplicate_counties.count())

display(duplicate_counties)

Save DimCounty to Silver

In [0]:
silver_county.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workforce_analytics.silver.dim_county")


saved_county = spark.table(
    "workforce_analytics.silver.dim_county"
)

print("Silver DimCounty rows:", saved_county.count())

display(saved_county)